### Import Libraries

In [3]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

import torch
import torchvision
from torchvision import transforms
import torch.nn as nn

# Моделі YOLOv8:
#  'yolov8n.pt' (nano) - найшвидша, найменш точна
#  'yolov8s.pt' (small) - швидка
#  'yolov8l.pt' (large) - точна, повільніша
#  'yolov8x.pt' (xlarge) - найточніша, найповільніша

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### YOLOv8

In [4]:
model = YOLO('yolov8l.pt')  # Завантаження попередньо навченого моделі YOLOv8s

class_names = ['car','bus','truck','bycle','motorcycle']  # Визначення класів об'єктів для виявлення

### Fine-tuning YOLOv8 on Custom Dataset

In [ ]:
model.train(
    data='data.yaml',  # Шлях до файлу з описом датасету
    epochs=50,         # Кількість епох для навчання
    batch=16,         # Розмір батчу
    imgsz=640,
    device=0,
    freeze=10,        # Розмір зображень
    name='yolov8l_custom',  # Ім'я експерименту для збереження результатів
)

Ultralytics 8.3.232 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8l_custom6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretraine

RuntimeError: Dataset 'data.yaml' error ❌ 'data.yaml' does not exist

In [ ]:
metrics = model.val() 
print(metrics)

### DeepSORT

In [7]:
import sys
sys.path.append(r'E:\PersonalProject\bachelor\YOLOv8-DeepSORT-Object-Tracking\ultralytics\yolo\v8\detect\deep_sort_pytorch')

In [8]:
from deep_sort.deep_sort import DeepSort

### Example of Object Tracking with YOLOv8 and DeepSORT (need to review and to extract usefull elements) 

In [ ]:
import cv2
from ultralytics import YOLO
# Initialize YOLO model
model = YOLO('yolov8n.pt')

# Initialize DeepSORT tracker
# Default embedder is mobileNetV2
tracker = DeepSort(max_age=30, n_init=3)

# Open video file
cap = cv2.VideoCapture('input_video.mp4')

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Run YOLOv8 detection on the frame
    results = model(frame)
    detections = []

    # Process detections for DeepSORT input format [x1, y1, x2, y2, confidence, class_name]
    for result in results:
        boxes = result.boxes
        for box in boxes:
            # Get bounding box coordinates in x1, y1, x2, y2 format
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]
            class_id = int(box.cls[0])
            class_name = model.names[class_id]
            
            # Filter detections if needed (e.g., only "person" class)
            if class_name == 'person' and confidence > 0.5:
                detections.append(([x1, y1, x2 - x1, y2 - y1], confidence, class_name)) # DeepSORT expects [x, y, w, h]

    # Update tracker with current detections
    tracks = tracker.update_tracks(detections, frame=frame)

    # Draw tracking results
    for track in tracks:
        if not track.is_confirmed():
            continue
        # Get track ID and bounding box
        track_id = track.track_id
        ltrb = track.to_ltrb()
        x1, y1, x2, y2 = map(int, ltrb)

        # Draw bounding box and ID
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display the frame
    cv2.imshow('YOLOv8 DeepSORT Tracking', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()